# Efecto disposición y sobreconfianza en un mercado simulado
## Reporte del proyecto P01 · Finanzas conductuales

El objetivo de este proyecto es estudiar si los indicadores utilizados para detectar sesgos de inversión reconocen los mecanismos que introducimos en un mercado artificial. Simulamos decisiones de compra y venta y comparamos ocho configuraciones que modifican el efecto disposición, la intensidad de negociación y dos explicaciones alternativas del comportamiento.

Los resultados muestran que PGR−PLR es cero en el escenario nulo, crece de forma monotónica con el parámetro de disposición y no reacciona a la frecuencia de operación. Sin embargo, el mismo indicador es positivo con rebalanceo y con reversión sin ninguna disposición, de modo que no identifica la motivación de las ventas. Además, la pendiente positiva del turnover sobre el rendimiento aparece incluso en el nulo por la forma de medir el turnover; con una definición normalizada desaparece.

**Alcance y trazabilidad.** Todas las cifras salen del notebook `P01_v2.ipynb` y de su módulo `p01_sim.py`, con `SEED_MUNDO=42` y `SEED_SIM=2026`; dos corridas completas son idénticas. Las tablas 4.1 a 4.3 corresponden a un mundo (población, precios y portafolio de la semilla 42) con números aleatorios comunes entre escenarios. La tabla 4.4 repite los ocho escenarios en 40 mundos con población, precios y portafolio regenerados (semillas 1000 a 1039).


## 1. Hipótesis y preanálisis

*Nota de transparencia: la versión 1 del simulador ya se había ejecutado; sus fallas (sin reinversión, generador aleatorio compartido, un solo costo agregado) motivaron la versión 2. Este párrafo fija las expectativas para la versión 2 antes de correr sus estimadores; no pretende ser un registro ciego de la primera ronda.*

Inyectaremos, para cada cuenta i, δᵢ = δs·uᵢ y κᵢ = κs·vᵢ, con uᵢ, vᵢ ~ U(0,1) independientes y δs, κs ∈ {0, 0.3, 0.8} según el escenario. La probabilidad diaria de vender un lote es 0.015·(1+1.5κᵢ)·(1±δᵢ), con + si está en ganancia y − si está en pérdida. Los escenarios 7 y 8 no usan δ ni κ: el 7 vende, cada 20 días, los lotes cuyo peso supera en más de 25% el peso objetivo 1/n; el 8 vende con probabilidad 1.6 veces la base si la acción subió en los últimos 20 días y 0.6 veces si bajó. La cartera neta paga comisión de 3 pb por lado y medio spread de 2 pb por lado. Esperamos que: (i) PGR−PLR sea ≈ 0 en los escenarios 1, 4 y 5 (el IC 95% por bootstrap de cuentas incluye el cero) y crezca de forma aproximadamente proporcional a δs en los escenarios 2, 3 y 6; (ii) el turnover crezca con κs y no dependa de δs en primer orden; (iii) como los precios son exógenos y las recompras aleatorias, ninguna regla altere el rendimiento bruto esperado, por lo que la pendiente del turnover sobre el rendimiento bruto debería ser cero y la del neto negativa y del tamaño del costo por unidad de turnover; anticipamos, sin embargo, que la definición original de turnover (ventas a precios corrientes entre capital inicial) contiene un componente mecánico que producirá una pendiente positiva, y que una versión normalizada por el valor medio de la cartera será neutral en el nulo; (iv) los escenarios 7 y 8 darán PGR−PLR > 0 aun con δ = 0, de modo que ese indicador no los separa de la disposición, pero una regresión de la probabilidad de venta sobre ganancia acumulada, rendimiento reciente y sobrepeso debería asignar cada patrón a su propio regresor.


## 2. Diseño del simulador
### Mercado y población
El mercado contiene 50 activos y 500 días de negociación. Los rendimientos siguen una estructura de un factor de mercado común: $r_{j,t}=\beta_j f_t+\varepsilon_{j,t}$, donde $f_t$ es un factor común a las 50 acciones y $\beta_j$ es la carga de cada acción a ese factor, sorteada independientemente por acción ($\beta_j\sim N(1.0,0.3^2)$, truncada en 0.2). Con esta calibración la volatilidad diaria por acción queda cerca de 1.5%, igual que en una versión sin factor, pero la correlación promedio entre pares de acciones queda alrededor de 0.44, del orden de lo observado en acciones reales. Los precios se generan antes de las decisiones y ninguna regla de venta consulta rendimientos futuros; esto no cambia por tener una estructura de factores, y se verifica directamente en la sección de confusores.

Se generan 1,000 cuentas, con capital inicial entre 10,000 y 500,000 dólares y entre 5 y 30 posiciones. El capital se distribuye inicialmente en partes iguales entre activos elegidos sin repetición. Se crean 16,833 posiciones y se permiten títulos fraccionarios.

### Parámetros y reglas de venta
Cada cuenta recibe dos sorteos independientes, $u_i$ y $v_i$, uniformes entre cero y uno. En cada escenario se utilizan:

$$\delta_i=\delta_s u_i,\qquad \kappa_i=\kappa_s v_i.$$

Por tanto, los valores 0.3 y 0.8 de las tablas son **factores de escala y límites superiores**, no valores idénticos para todas las cuentas. Con escala 0.8, las medias realizadas son aproximadamente 0.3977 para disposición y 0.4068 para frecuencia.

Los escenarios usan tres reglas de venta:

- **Base** (escenarios 1 a 6). La probabilidad diaria de vender una posición ganadora o perdedora es
$$p_{i,+}=0.015(1+1.5\kappa_i)(1+\delta_i),\qquad p_{i,-}=0.015(1+1.5\kappa_i)(1-\delta_i).$$
- **Rebalanceo** (escenario 7). Probabilidad base neutral, $0.015$, sin depender de ganancia o pérdida. Además, cada 20 días se venden los lotes cuyo peso en la cartera supera en más de 25% el peso objetivo $1/n$; el lote vendido se reemplaza por uno de tamaño objetivo y el excedente se reparte entre las demás posiciones a costo promedio. No consulta el precio de compra.
- **Reversión** (escenario 8). La probabilidad base se multiplica por 1.6 si la acción subió en los últimos 20 días y por 0.6 si bajó. No consulta el precio de compra.

Los parámetros actúan sobre la salida diaria de cada posición. PGR y PLR se obtienen después, mediante conteos en días con ventas, y no se fijan directamente. Una respuesta monotónica de PGR−PLR valida la dirección del mecanismo, pero no significa que ese indicador recupere numéricamente $\delta_i$.

### Reinversión, costos y turnover
Cada posición vendida se sustituye inmediatamente por un activo elegido al azar al precio del mismo día, de modo que se conserva el número de lotes y la cartera nunca mantiene efectivo. El simulador mantiene dos carteras paralelas con las mismas decisiones: una bruta, sin fricciones, y una neta, que paga costos en cada venta y en cada recompra:

- **Comisión:** 0.03% del monto operado, por lado.
- **Spread:** 0.04% en total; se paga la mitad (0.02%) en cada lado, porque se vende al bid y se compra al ask.

Cada lado cuesta 0.05% y la ida y vuelta 0.10% (10 puntos base). Los valores son supuestos del ejercicio. Comisiones y spreads se acumulan por separado.

El turnover se reporta con tres definiciones: (a) la original, ventas de la cartera bruta a precios corrientes divididas entre el capital inicial; (b) ventas divididas entre el valor medio de la cartera; y (c) operaciones por posición. Los rendimientos son acumulados durante los 500 días:

$$Turnover_i=\frac{\sum_t V^{\mathrm{venta,bruta}}_{i,t}}{C_{i,0}},\qquad Turnover^{norm}_i=\frac{\sum_t V^{\mathrm{venta,bruta}}_{i,t}}{\bar V_i},\qquad R_i=\frac{V_{i,T}-C_{i,0}}{C_{i,0}}.$$

### Aleatoriedad y comparabilidad
Cada simulación recibe una semilla explícita y no hay generador global. Dentro de un mismo mundo los ocho escenarios usan los mismos números aleatorios (uniformes de venta y sorteos de recompra), por lo que sus diferencias se deben al mecanismo y no al ruido de Monte Carlo. Para medir la variabilidad entre mercados, el experimento completo se repite en 40 mundos con población, precios y portafolio regenerados.


## 3. Estimación y convenciones
En cada día en que una cuenta vende algo se cuentan sus ganancias realizadas ($G_r$), ganancias conservadas ($G_p$), pérdidas realizadas ($L_r$) y pérdidas conservadas ($L_p$). Los indicadores son:

$$PGR=\frac{G_r}{G_r+G_p},\qquad PLR=\frac{L_r}{L_r+L_p},\qquad D=PGR-PLR,\qquad Q=\frac{PGR}{PLR}.$$

Se aplican las siguientes convenciones:

- Los días sin ventas de una cuenta no contribuyen a sus conteos.
- No hay ventas parciales: cada lote se vende completo y cuenta como una realización.
- La base de costo es el precio de compra de cada lote y se actualiza al recomprar. Los lotes de una misma emisora no se consolidan, salvo los lotes que reciben excedente en el rebalanceo, que pasan a costo promedio.
- Una posición exactamente a su precio de compra se excluye de ganancias y pérdidas (en la regla base tiene probabilidad de venta cero).
- Las posiciones conservadas se contabilizan antes de incorporar las recompras del día.

**Errores estándar.** El error estándar de $D$ y $Q$ se obtiene remuestreando cuentas completas (1,000 réplicas, conservando juntos los cuatro conteos de cada cuenta). La variabilidad entre mercados se estima con los 40 mundos regenerados.

**Regresiones.** Se usa una observación por cuenta y errores robustos HC1. El modelo incluye la medida de turnover, capital inicial en miles de dólares, número de posiciones, la **volatilidad diaria realizada** de la cartera bruta como medida de exposición al riesgo y los parámetros individuales que varían en cada escenario. La volatilidad realizada tiene correlación de aproximadamente 0.98 con $1/\sqrt{n}$, de modo que el número de posiciones ya funcionaba como aproximación de la exposición. Se estiman tres definiciones de turnover en los ocho escenarios.

**Panel lote-día.** Para separar mecanismos se usa un panel de 200 cuentas, un día de cada cinco (336,400 observaciones). La variable dependiente es vender o no el lote ese día y los regresores son ganar sobre el costo, rendimiento de 20 días positivo y sobrepeso. Se estima un modelo lineal de probabilidad con errores agrupados por cuenta.


## 4. Resultados
### 4.1. Disposición y precisión de las estimaciones

| Escenario | δs / κs (o regla) | PGR | PLR | D | EE(D) | Q | EE(Q) |
|---|---|---:|---:|---:|---:|---:|---:|
| 1. Nulo | 0.0 / 0.0 | 0.0580 | 0.0578 | 0.00021 | 0.00033 | 1.0037 | 0.0057 |
| 2. Disposición baja | 0.3 / 0.0 | 0.0648 | 0.0487 | 0.01608 | 0.00049 | 1.3301 | 0.0112 |
| 3. Disposición alta | 0.8 / 0.0 | 0.0783 | 0.0342 | 0.04404 | 0.00108 | 2.2873 | 0.0462 |
| 4. Frecuencia baja | 0.0 / 0.3 | 0.0601 | 0.0601 | 0.00005 | 0.00030 | 1.0008 | 0.0050 |
| 5. Frecuencia alta | 0.0 / 0.8 | 0.0641 | 0.0641 | 0.00000 | 0.00029 | 1.0000 | 0.0045 |
| 6. Ambos activos | 0.8 / 0.8 | 0.0875 | 0.0386 | 0.04892 | 0.00122 | 2.2677 | 0.0461 |
| 7. Rebalanceo | rebalanceo | 0.0644 | 0.0547 | 0.00977 | 0.00034 | 1.1788 | 0.0065 |
| 8. Reversión | reversion | 0.0695 | 0.0464 | 0.02304 | 0.00036 | 1.4964 | 0.0081 |

EE significa error estándar (bootstrap de cuentas). D y Q se calculan con conteos sin redondear. Los escenarios 7 y 8 no usan δ ni κ: se identifican por su regla.


### 4.2. Actividad y rendimientos acumulados

| Escenario | Turnover (orig.) | Turnover norm. | Ops/posición | Bruto | Neto | Brecha (pp) | Comisiones (USD) | Spreads (USD) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| 1. Nulo | 8.787 | 7.518 | 7.505 | 42.37% | 41.31% | 1.065 | 1,317 | 878 |
| 2. Disposición baja | 9.081 | 7.759 | 7.653 | 42.76% | 41.66% | 1.100 | 1,360 | 907 |
| 3. Disposición alta | 9.102 | 7.751 | 7.473 | 44.00% | 42.89% | 1.104 | 1,365 | 910 |
| 4. Frecuencia baja | 10.800 | 9.237 | 9.226 | 42.50% | 41.19% | 1.309 | 1,616 | 1,077 |
| 5. Frecuencia alta | 14.166 | 12.116 | 12.097 | 42.61% | 40.90% | 1.715 | 2,112 | 1,408 |
| 6. Ambos activos | 14.139 | 12.087 | 11.713 | 43.31% | 41.60% | 1.713 | 2,106 | 1,404 |
| 7. Rebalanceo | 10.097 | 8.616 | 8.338 | 43.80% | 42.58% | 1.227 | 1,513 | 1,008 |
| 8. Reversión | 10.291 | 8.812 | 8.615 | 42.73% | 41.49% | 1.248 | 1,541 | 1,027 |

La brecha bruto−neto incluye el efecto acumulado de reinvertir menos capital después de cada costo; no equivale a los costos pagados divididos entre capital inicial. «pp» indica puntos porcentuales. El rendimiento del mundo 42 (42.4% en el nulo) es superior al esperado; ver 4.4.


### 4.3. Pendiente del turnover por escenario

| Escenario | β bruto, turnover orig. (EE) | β neto (EE) | β bruto, turnover norm. (EE) | β neto (EE) |
|---|---:|---:|---:|---:|
| 1. Nulo | 0.0480 (0.0045) | 0.0467 (0.0045) | -0.0030 (0.0060) | -0.0044 (0.0060) |
| 2. Disposición baja | 0.0576 (0.0040) | 0.0562 (0.0040) | 0.0124 (0.0062) | 0.0109 (0.0062) |
| 3. Disposición alta | 0.0618 (0.0035) | 0.0605 (0.0035) | 0.0341 (0.0060) | 0.0325 (0.0060) |
| 4. Frecuencia baja | 0.0448 (0.0041) | 0.0434 (0.0041) | -0.0049 (0.0052) | -0.0062 (0.0052) |
| 5. Frecuencia alta | 0.0424 (0.0030) | 0.0410 (0.0029) | -0.0029 (0.0047) | -0.0043 (0.0046) |
| 6. Ambos activos | 0.0457 (0.0021) | 0.0443 (0.0021) | 0.0285 (0.0037) | 0.0268 (0.0037) |
| 7. Rebalanceo | 0.0469 (0.0039) | 0.0455 (0.0039) | -0.0050 (0.0054) | -0.0064 (0.0054) |
| 8. Reversión | 0.0585 (0.0033) | 0.0571 (0.0033) | 0.0271 (0.0055) | 0.0255 (0.0054) |

Los coeficientes expresan cambios en rendimiento decimal por una unidad de turnover; entre paréntesis, error estándar robusto HC1. Controles: capital, número de posiciones, volatilidad realizada y parámetros individuales no constantes.


### 4.4. Incertidumbre entre 40 mercados

| Escenario | D medio | SD(D) entre mundos | EE(D) por cuentas | Bruto medio | SD bruto entre mundos | EE bruto por cuentas | β turnover orig. | β turnover norm. |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| 1. Nulo | 0.00008 | 0.00029 | 0.00032 | 28.6% | 5.63 pp | 0.37 pp | 0.0490 | -0.0004 |
| 2. Disposición baja | 0.01615 | 0.00052 | 0.00048 | 28.6% | 5.65 pp | 0.37 pp | 0.0553 | 0.0145 |
| 3. Disposición alta | 0.04383 | 0.00115 | 0.00109 | 28.5% | 5.73 pp | 0.37 pp | 0.0598 | 0.0380 |
| 4. Frecuencia baja | 0.00006 | 0.00027 | 0.00030 | 28.6% | 5.59 pp | 0.37 pp | 0.0470 | -0.0001 |
| 5. Frecuencia alta | -0.00002 | 0.00029 | 0.00027 | 28.6% | 5.58 pp | 0.37 pp | 0.0429 | -0.0005 |
| 6. Ambos activos | 0.04866 | 0.00140 | 0.00121 | 28.5% | 5.72 pp | 0.37 pp | 0.0451 | 0.0281 |
| 7. Rebalanceo | 0.01018 | 0.00084 | 0.00032 | 28.5% | 5.60 pp | 0.37 pp | 0.0517 | 0.0066 |
| 8. Reversión | 0.02313 | 0.00057 | 0.00036 | 28.5% | 5.73 pp | 0.37 pp | 0.0565 | 0.0245 |

Cada mundo regenera población, precios y portafolio. SD(D) es la desviación estándar de $D$ entre mundos; se compara con el error estándar por cuentas. El rendimiento esperado con estos parámetros es $(1.0005)^{500}-1=28.4\%$.


### 4.5. Panel lote-día: qué variable explica cada venta

| Escenario | Ganancia sobre el costo | Rend. 20 días > 0 | Sobrepeso |
|---|---:|---:|---:|
| 1. Nulo | −0.007 (t −0.1) | 0.005 (t 0.1) | 0.026 (t 0.4) |
| 3. Disposición alta | **1.198** (t 17.8) | −0.034 (t −0.6) | 0.050 (t 0.7) |
| 7. Rebalanceo | −0.031 (t −0.6) | 0.014 (t 0.3) | **32.884** (t 86.8) |
| 8. Reversión | 0.039 (t 0.9) | **1.399** (t 30.5) | 0.063 (t 0.9) |

Coeficientes en puntos porcentuales de probabilidad diaria de venta; entre paréntesis, estadístico t con errores agrupados por cuenta.


## 5. Interpretación de los resultados
### Validación

**Recuperación del nulo.** D = 0.0002 con EE 0.0003. En 40 mercados nuevos, D del nulo tiene media 0.0001 y desviación 0.0003; el z = D/EE tiene desviación 0.93 y rechaza |z|>1.96 en 2.5% de los mercados (esperado ≈ 5%). Con κ solo rechaza en 5.0% (ambos escenarios).

**Monotonicidad.** Con malla fina (10 mercados, δs de 0 a 1) D = 0.0001, 0.0108, 0.0216, 0.0325, 0.0439, 0.0558: estrictamente creciente y casi lineal (≈ 0.055·δs). Con κs de 0 a 1 el turnover normalizado sube de 7.50 a 13.11 y D queda en cero. En 40 de 40 mercados D(1) < D(2) < D(3) y turnover(1) < turnover(4) < turnover(5).

**Estimadores cruzados.** El indicador de disposición permanece en cero cuando solo se mueve κ. La pendiente del turnover normalizado también es cero con κ solo (−0.005 y −0.003; EE 0.005), pero no lo es con δ (0.012 y 0.034): la disposición contamina ese estimador por causalidad inversa (sección 6). La validación cruzada se cumple en un sentido y falla en el otro. Con δ activo, el turnover normalizado incluso baja para δs > 0.4 (7.59 con δs=0.4 a 6.97 con δs=1.0), coherente con que retener perdedoras reduce las ventas.

### Costos e interacción entre mecanismos
Con frecuencia alta (escenario 5) el rendimiento bruto medio es 42.61%, frente a 42.37% en el nulo, mientras que el neto baja de 41.31% a 40.90%. La brecha bruto−neto pasa de 1.065 a 1.715 puntos porcentuales; las comisiones medias por cuenta suben de 1,317 a 2,112 dólares y los spreads de 878 a 1,408. La razón spread/comisión se mantiene en 0.67, igual a 0.02%/0.03%, como corresponde al diseño. En 40 mercados el bruto medio es 28.5%–28.6% en todos los escenarios: los mecanismos no alteran el rendimiento bruto esperado y las diferencias dentro de un mercado son ruido de decisión.

Cuando ambos parámetros están activos, D = 0.0489, ligeramente mayor que con disposición sola (0.0440). Esa diferencia (+0.005) aparece en los 40 mercados, de modo que no es ruido, pero su origen no se investigó. La presencia conjunta no demuestra que los mecanismos sean aditivos.

### Confusores: una señal positiva admite varias explicaciones
Cada confusor decide con una variable distinta de la que usa la disposición y no consulta el precio de compra (rebalanceo: peso; reversión: rendimiento de 20 días). Si se hubieran definido como multiplicadores sobre la ganancia frente al precio de compra, D > 0 sería inevitable y no informaría nada. Ambos producen D > 0 con δ = 0: 0.0098 (EE 0.0003) y 0.0230 (EE 0.0004), frente a 0.0440 con disposición alta. El indicador capta la correlación entre la ganancia sobre el costo y esas otras variables, de modo que **no identifica la motivación**: una D positiva es compatible con disposición, rebalanceo o reversión, y la disposición baja (0.0161) produce más señal que el rebalanceo.

Con las variables correctas, el panel lote-día de la tabla 4.5 (200 cuentas, un día de cada cinco; 336,400 observaciones; modelo lineal de probabilidad con errores agrupados por cuenta) sí los separa (tabla 4.5). Disposición: la ganancia acumulada eleva la probabilidad diaria de vender en 1.20 pp (t = 17.8) y el rendimiento reciente y el sobrepeso no importan (≈ 0). Reversión: el rendimiento positivo de 20 días eleva la probabilidad en 1.40 pp (t = 30.5) y los demás regresores no importan. Rebalanceo: el sobrepeso la eleva en 32.9 pp (t = 86.8) y los demás no importan. En el nulo los tres son cero.

Este resultado depende de observar sin error las variables que generan las decisiones. Con datos reales no se observan pesos objetivo, pronósticos ni expectativas, y ahí el indicador PGR−PLR por sí solo no basta: por eso la tabla de datos necesarios de la sección 9. Limitaciones: la reversión es una regla sobre el rendimiento pasado en un mercado sin reversión real, y el rebalanceo es por bandas cada 20 días; ninguno modela expectativas explícitas.


## 6. Por qué la pendiente de turnover sale positiva

De las cinco posibles causas de una pendiente negativa entre turnover y rendimiento bruto, descartamos tres por diseño y verificamos las otras dos directamente:

| Causa | Estado | Evidencia |
|---|---|---|
| Fuga de información hacia la decisión de compra | Descartada, verificada | Rendimiento a 20 días de lo vendido vs. una recompra aleatoria del mismo día: diferencia entre +0.00002 y +0.00046 en los escenarios probados, sobre un rendimiento a 20 días de ≈1.5% — indistinguible de cero. |
| Cash drag | No aplica por diseño | Cada venta se reemplaza en la misma operación; la cartera nunca sostiene efectivo. |
| Spread dentro del precio | No aplica por diseño | La cartera bruta se simula sin comisión ni spread; el rendimiento bruto no lleva costos descontados. |
| Efectos de composición geométrica (compounding) | Descartada, verificada | En los escenarios con solo κ activo (δ=0), el coeficiente de κᵢ sobre el rendimiento bruto, sin turnover en la regresión, no es significativo (t=−1.30 y −1.51 en frecuencia baja y alta). |
| Causalidad inversa | Confirmada | Ya documentada: la pendiente normalizada solo se vuelve positiva cuando δ está activo, y desaparece controlando por turnover. |

La única causa presente en este diseño es la causalidad inversa, ya caracterizada en la sección anterior. Las demás se descartan por construcción del simulador o por verificación directa, no por supuesto.


## 7. Independencia de parámetros e interacción observada
La correlación de los sorteos individuales de disposición y frecuencia es **0.00163**. En el escenario 6 se conserva ese valor porque ambos se multiplican por una constante positiva. Es compatible con el diseño de sorteos independientes.

Sin embargo, la independencia de los parámetros no implica independencia entre un parámetro y el comportamiento que produce:

| Escenario | Correlación δi–κi | Correlación δi–turnover | Correlación δi–turnover normalizado |
|---|---:|---:|---:|
| 2. Disposición baja | No definida: κi constante | 0.1483 | 0.1725 |
| 3. Disposición alta | No definida: κi constante | −0.0323 | −0.0587 |
| 6. Ambos activos | 0.0016 | −0.1263 | −0.1386 |

En los demás escenarios, δi es constante en cero y su correlación con turnover no está definida; no debe reportarse como cero. El signo cambia entre escenarios: es el resultado neto de vender más ganadoras y retener perdedoras. Las correlaciones resumen el balance realizado de estos mecanismos; no identifican por sí mismas sus contribuciones causales.

Si disposición y frecuencia se hubieran correlacionado por construcción, una cuenta con mayor intensidad de operación también tendería a tener otra intensidad de disposición, y sería difícil atribuir la relación entre turnover y rendimiento a un solo mecanismo. Sortear los parámetros independientemente evita esa asociación inicial, aunque no elimina la interacción de sus efectos.


## 8. Errores estándar, clustering e incertidumbre entre mercados

El error estándar de D se obtiene remuestreando cuentas completas (1,000 réplicas, conservando juntos los cuatro conteos de cada cuenta). Se contrasta con la desviación de D entre 40 mercados con población, precios y portafolio regenerados. En los escenarios base coinciden (nulo: 0.00032 por cuentas frente a 0.00029 entre mercados; disposición alta: 0.00109 frente a 0.00115). No coinciden en los confusores, cuyo D depende de la trayectoria de precios: rebalanceo 0.00032 frente a 0.00084 (2.6 veces) y reversión 0.00036 frente a 0.00057 (1.6 veces). Para esos escenarios el bootstrap de cuentas subestima la incertidumbre.

Con el rendimiento la diferencia es mucho mayor: el error estándar por cuentas es 0.37 pp, pero entre mercados la desviación estándar es 5.6 pp. El mercado de la semilla 42 dio 42.4% de rendimiento bruto medio, unas 2.4 desviaciones sobre el promedio de los 40 mercados (28.6%, cercano al 28.4% teórico). Por eso los niveles de rendimiento de un solo mercado no se interpretan como estimaciones del mercado, y las comparaciones entre escenarios sólo son válidas condicionalmente al mundo, gracias a los números aleatorios comunes.

Las regresiones usan una observación por cuenta con errores robustos HC1. El panel lote-día de la tabla 4.5 usa errores agrupados por cuenta, porque un mismo lote aparece en muchos días. No se agrupa por acción: cada cuenta mantiene varias acciones y no hay anidamiento; la dependencia entre cuentas proviene de la trayectoria común de precios y se cubre repitiendo el mercado.


## 9. Límites de identificación y datos necesarios
| Lo que los indicadores no distinguen | Información que permitiría avanzar |
|---|---|
| Disposición frente a rebalanceo y a reversión (mismo D > 0) | Peso de cada posición respecto al objetivo y rendimiento reciente por lote-día, además del precio de compra; con esos tres regresores la simulación los separa (tabla 4.5). |
| Disposición frente a expectativas de reversión reales | Pronósticos registrados antes de operar, señales utilizadas y resultados posteriores de activos comprados y vendidos. |
| Sobreconfianza frente a otras razones de negociación | Medidas de confianza y creencias, necesidades de liquidez, horizonte y restricciones por cuenta. |
| Efectos de costos frente a efectos de exposición | Precios de ejecución observados, exposición al riesgo y composición de cartera por fecha. |
| Turnover como causa o consecuencia del rendimiento | Historias temporales de carteras y operaciones, medidas normalizadas de turnover y experimentos que varíen la frecuencia de forma controlada. |

La simulación permite conocer el mecanismo que genera los datos, pero esa información no se recupera automáticamente con un indicador agregado. Este es precisamente el valor de incluir los escenarios alternativos y revisar los resultados que se apartan de las hipótesis.


## 10. Conclusiones

1. Con el mecanismo δ inyectado, PGR−PLR responde de forma monotónica y casi lineal (≈ 0.055·δs), el nulo es compatible con cero y la sobreconfianza (κ) no genera una falsa disposición. Su error estándar por cuentas es fiable en los escenarios base (desviación entre 40 mercados 0.0003, igual al EE del nulo).
2. El mismo indicador es positivo con rebalanceo (0.0098) y con reversión (0.0230) sin ninguna disposición: una D positiva no identifica la motivación. Con datos de peso y rendimiento reciente, un panel lote-día sí separa los tres mecanismos.
3. La pendiente negativa esperada del turnover sobre el rendimiento neto no aparece con la definición original: en el nulo la pendiente es +0.048 y positiva en 40 de 40 mercados, por cómo se mide el turnover. Normalizado, es cero (−0.003, EE 0.006) y el efecto de costos (≈ −0.0013 por unidad) se recupera con precisión. Con disposición, la pendiente normalizada es positiva (0.012 a 0.034) por causalidad inversa: quien gana más vende más.

El resultado más importante es que una simulación puede generar el patrón esperado en un indicador y, al mismo tiempo, revelar problemas de identificación en otro. Reconocer esa diferencia permite evaluar el modelo con más rigor y evita confundir un resultado estadístico con una explicación del comportamiento.

### Limitaciones

- Las tablas principales provienen de un solo mercado, cuyo rendimiento (+42.4%) es unas 2.4 desviaciones superior al promedio de 40 mercados; los niveles de rendimiento no son estimaciones del mercado, y las comparaciones entre escenarios son condicionales al mundo.
- Los valores de comisión y spread son supuestos.
- Rebalanceo y reversión son reglas reducidas (bandas de peso cada 20 días; rendimiento de 20 días), sin expectativas ni pesos objetivo heterogéneos; el mercado no tiene reversión real.
- La sobreconfianza se modela solo como mayor frecuencia de operación.
- Existe una pequeña interacción δ×κ (+0.005 en D) cuyo origen no se investigó.
- El panel lote-día usa 200 cuentas y un día de cada cinco.


### Nota de reproducción
Todas las cifras se generan con `P01_v2.ipynb` y `p01_sim.py` (*Entorno → Reiniciar y ejecutar todo*), con `SEED_MUNDO=42`, `SEED_SIM=2026`, semillas 1000 a 1039 para los 40 mundos y 2000 a 2009 para la malla de monotonicidad. No hay estado global ni dependencia del orden de las celdas, y una segunda corrida completa verifica igualdad exacta. Probado con Python 3.12 y statsmodels 0.15 en dos combinaciones de librerías (numpy 2.4 con pandas 3.0, y numpy 2.0 con pandas 2.2) con resultados idénticos. Repositorio: <URL de GitHub>.
